# 08 — Kiểm tra trực quan QRS/T trên toàn bộ448 recordings

**Tách T bằng Wavelet/TLS đã được bổ sung ở [mục 14C–14F](#nb08-wavelet-tls):**
gallery V1/V2 cho đủ 448 ID, các bước WT → single-lead → TLS → T-on/T-off,
hai T loops 3D và các ca chưa giải được. **Mục 14F nhúng hình của tất cả 448 ID**,
mỗi ca có XYZ/biên T và T loops V1/V2. Có thể đổi `ALMEIDA_RECORD_ID` ở mục 14D
để xem chi tiết các bước tính của bản ghi khác.

**Luồng chính:** mở gallery → chọn recording → xem median Frank XYZ và bốn mốc
V1 đã chọn → zoom QRS/T → xem loops3D → tự đặt mốc → so V1 với mốc thủ công
→ lưu → Previous/Next. Không cần đổi RECORD_ID hoặc chạy lại notebook mỗi lần.

Dùng trực tiếp `vx/vy/vz` của PTB, median đặt gốc PQ, đúng quy trình notebook05
Bước6–19. Màu xanh nhạt là khoảng QRS mà V1 đã cắt; cam nhạt là khoảng T.
Tín hiệu ngoài các vùng tô vẫn hiển thị để kiểm tra phần đầu/cuối có bị bỏ sót.
Candidate tangent/threshold là lớp tham khảo, **không chứng minh biên đúng**.

**14A đã nâng cấp:** gallery offline bao phủ đủ448 ca, chuyển từng ca, lọc/tìm,
zoom, bật/tắt biên và lưu Manual Visual Review. **14B đã nâng cấp:** toàn bộ
contact sheets là XYZ thật với biên cuối cùng V1, không còn lấy chênh candidate làm nội dung chính.

Manual Visual Review có thể đã xem prediction: lưu riêng, luôn
`independent_reference=False`; không ghi vào annotation độc lập hoặc thay prediction.
Phân tích reference/sai số cũ được giữ ở **phụ lục**, chỉ tính với reference được chấp nhận.

Kernel **ipcv** → **Run All**. Các hình/payload có cache. Đầu ra trực quan ở
`outputs/vcg_boundary_reference_eval_v1/all_cases/`; không sửa notebooks05/06/07,
dữ liệu V1 hoặc split theo bệnh nhân. Test chỉ được xem mô tả, không tinh chỉnh.

## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(8)

## 1. Import, cấu hình và giữ nguyên V1

Giữ đường dẫn annotation độc lập của người dùng. Các tùy chọn bổ sung không thay đổi prediction.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, IFrame

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'vcg_pipeline.py').is_file())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT/'scripts'))
from vcg_pipeline import FEATURE_NAMES
from vcg_boundary_reference_eval import (initialize, scan_reference_sources, priority_cases,
    import_notebook07_reviews, read_annotations, save_results, write_report,
    verify_protected, snapshot_sources, write_json, sha256)
from vcg_reference_metrics import evaluate_references, load_median
from vcg_reference_annotation_ui import build_annotation_tool
from vcg_reference_plots import create_reference_plots, create_diagnostic_case_gallery

# Thêm CSV đã được reviewer export; không dùng các mốc prediction làm ref.
ANNOTATION_FILES = []
FINAL_TEST_EVALUATION = False
OUT = ROOT/'outputs/vcg_boundary_reference_eval_v1'
frame, manifest, CFG, ARTIFACTS, PROTECTED = initialize(OUT, allow_test=FINAL_TEST_EVALUATION)
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.max_rows', 40)
print('Read-only V1; output:', OUT)
print('Frank channels: vx, vy, vz | Hệ thời gian: median_relative_ms')
print('Test reference evaluation:', FINAL_TEST_EVALUATION)

SHOW_METHOD_DIAGNOSTICS = False  # Phụ lục tùy chọn; không phải nội dung xem chính.
from vcg_visual_review_data import prepare_visual_review_data
from vcg_visual_review_ui import build_visual_review_gallery
from vcg_visual_review_overview import create_v1_contact_sheets
from vcg_boundary_disagreement_panels import validate_case_panels

## 2. Dữ liệu đúng notebook05, median và mốc V1 đúng notebook06

Kiểm tra đầy đủ448ID; thời gian mỗi ca theo fs/anchor riêng. Tính lại intermediate bằng đúng delineate_loops để kiểm tra nhất quán; mọi khác biệt được ghi, không thay mốc đã lưu.

In [ ]:
REVIEW_OUT = OUT/'all_cases'
DATA = prepare_visual_review_data(frame,ARTIFACTS,REVIEW_OUT,cfg=CFG,resume=True)
signal_manifest = DATA['manifest']
assert len(signal_manifest)==448 and signal_manifest.record_id.is_unique
data_metrics = {key:DATA['validation'][key] for key in [
    'n_records','n_medians','n_qrs_loops','n_t_loops','n_parity_fields',
    'n_parity_differences','n_integrity_issues','cache_hits']}
display(pd.DataFrame([data_metrics]))
display(signal_manifest.groupby(['label_name','split','segmentation_status']).size().rename('recordings').to_frame())
differences=DATA['parity'][~DATA['parity'].matches_exactly.astype(bool)]
if len(differences):
    display(differences)
else:
    print('Toàn bộ intermediate và mốc V1 đối chiếu đều khớp; không thay prediction.')
display(Markdown('**Nguồn hiển thị:** XYZ đã trừ baseline PQ và biên/sample đã lưu. '
    't[i]=(i−anchor)×1000/fs. QRS và T được cắt riêng bằng slice inclusive; không lấy đoạn ST giữa hai loop. '
    'Các payload lỗi/thiếu vẫn giữ nguyên ID để xuất hiện trong giao diện.'))

## 14A. Gallery tương tác: xem từng recording, zoom và Manual Visual Review

Đây là giao diện chính cho toàn bộ448 ca, không lọc theo QC hoặc số candidate. A: full median XYZ; B: QRS XYZ+velocity/ngưỡng/search; C: T XYZ+magnitude/ST/TP; D: hai loop V1 trong hệ PQ, cùng tỉ lệ, kèm hướng thời gian và projections. Những mốc/loop không tồn tại được báo thiếu.

In [ ]:
import subprocess
rendered = subprocess.run([sys.executable,str(ROOT/'scripts/vcg_boundary_disagreement_panels.py'),
    '--output-dir',str(REVIEW_OUT),'--workers','4'],capture_output=True,text=True)
if rendered.returncode:
    raise RuntimeError(rendered.stdout+chr(10)+rendered.stderr)
panel_manifest = pd.read_csv(REVIEW_OUT/'all_case_panel_manifest.csv')
panel_checks = validate_case_panels(panel_manifest,REVIEW_OUT,frame.record_id.tolist())
gallery = build_visual_review_gallery(frame,signal_manifest,panel_manifest,REVIEW_OUT)
print('Gallery:',gallery['gallery_path'])
display(pd.DataFrame([panel_checks]))
display(Markdown('[Mở gallery offline toàn448 ca](outputs/vcg_boundary_reference_eval_v1/all_cases/all_cases_gallery.html). '
    'Dùng Previous/Next hoặc danh sách ID; bộ lọc vẫn giữ các ca thiếu biên/median. '
    'Nút tổng quan chuyển sang danh sách hình nhiều ca. Chế độ chi tiết chỉ tải signal của ca đang chọn.'))
display(IFrame(src='outputs/vcg_boundary_reference_eval_v1/all_cases/all_cases_gallery.html',width='100%',height=1250))

<a id="nb08-wavelet-tls"></a>

## 14C. T loops mới: gallery Wavelet/TLS và đối chiếu V1/V2

Phần này đưa trực tiếp kết quả thí nghiệm Notebook 09 vào Notebook 08.
Mở gallery bên dưới để tìm ID, chuyển Previous/Next, lọc nhãn/split/trạng thái,
xem XYZ, WT ở scale 3/4/5, single-lead candidates, từng vòng TLS và T loops 3D.
Gallery giữ đủ **448 ID**, kể cả những ca thiếu median hoặc thiếu biên T.

Dữ liệu hiển thị lấy từ kết quả đã lưu của cấu hình Wavelet/TLS hiện tại.
QRS, median Frank XYZ, gốc PQ và hệ thời gian vẫn theo V1. Các biên T V1 chỉ
dùng đối chiếu sau khi V2 đã được tính độc lập; phần này không tính lại V1.

Đây là bản dựng theo công thức Almeida/Martínez có điều chỉnh đã công bố,
**chưa xác minh tương đương hoàn toàn mã tác giả**. Xem
[Notebook 09](09_VCG_T_Loop_Almeida_Wavelet_TLS_Experiment.ipynb) và
[truy vết phương pháp](METHOD_REPRODUCTION.md) để đọc nguồn và giới hạn.
Chênh V2−V1 là bất đồng phương pháp; chưa có reference độc lập được chấp nhận
để kết luận cách nào chính xác hơn.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, IFrame

almeida_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                   if (p/'vcg_t_almeida.py').is_file())
if str(almeida_root/'scripts') not in sys.path:
    sys.path.insert(0, str(almeida_root/'scripts'))
if str(almeida_root) not in sys.path:
    sys.path.insert(0, str(almeida_root))
almeida_out = almeida_root/'outputs/vcg_t_almeida_v2_experiment'
almeida_required = ['v2_t_boundaries.csv', 'split_manifest.csv', 'config.json',
                    'execution_summary.json', 'gallery/index.html', 'gallery/manifest.csv']
almeida_missing = [p for p in almeida_required if not (almeida_out/p).is_file()]
if almeida_missing:
    raise FileNotFoundError('Thiếu kết quả Notebook 09: '+', '.join(almeida_missing)+
        '. Chạy scripts/run_vcg_t_almeida_experiment.py trước.')
almeida_results = pd.read_csv(almeida_out/'v2_t_boundaries.csv')
almeida_manifest = pd.read_csv(almeida_out/'split_manifest.csv')
almeida_gallery_manifest = pd.read_csv(almeida_out/'gallery/manifest.csv')
almeida_config = json.loads((almeida_out/'config.json').read_text())
almeida_summary = json.loads((almeida_out/'execution_summary.json').read_text())
almeida_coverage = pd.read_csv(almeida_out/'reports/coverage_by_label_and_split.csv')
if (not almeida_results.record_id.is_unique or len(almeida_results) != 448
        or set(almeida_results.record_id) != set(almeida_manifest.record_id)
        or set(almeida_results.record_id) != set(almeida_gallery_manifest.record_id)):
    raise ValueError('Kết quả/gallery V2 không khớp đầy đủ manifest 448 ID.')
if not almeida_results.algorithm_fingerprint.eq(almeida_config['algorithm_fingerprint']).all():
    raise ValueError('Kết quả V2 khác cấu hình đã lưu; không trộn các lần chạy.')
display(pd.Series({
    'recordings': len(almeida_results),
    'median_available': int(almeida_results.median_present.sum()),
    'T_on_V2_available': int(almeida_results.t_on_v2_ms.notna().sum()),
    'T_off_V2_available': int(almeida_results.t_off_v2_ms.notna().sum()),
    'T_loops_V2': int(almeida_results[['t_on_v2_ms','t_off_v2_ms']].notna().all(axis=1).sum()),
    'unresolved': int(almeida_results.status.eq('unresolved').sum()),
}, name='Số bản ghi').to_frame())
display(almeida_coverage)
display(Markdown(
    '[Mở gallery Wavelet/TLS V1–V2 toàn bộ 448 ID trong tab riêng]'
    '(outputs/vcg_t_almeida_v2_experiment/gallery/index.html). '
    'Các tab: Signals → Wavelets → Single-lead → TLS → Loops → History. '
    'Trong tab TLS chọn nhánh T-on/T-off và iteration; kéo hình 3D để xoay. '
    'Nếu iframe bị chặn trong trình xem notebook, dùng liên kết mở tab riêng.'))
display(IFrame(src='outputs/vcg_t_almeida_v2_experiment/gallery/index.html',
               width='100%', height=1120))

## 14D. Xem trực tiếp từng bước tách T bằng Wavelet/TLS

Đổi `ALMEIDA_RECORD_ID` ở cell dưới, rồi chạy từ đây tới hết mục 14D.
Mặc định dùng `patient104/s0306lre`, cũng là walkthrough của Notebook 09.
Muốn chuyển liên tục giữa mọi bản ghi mà không chạy lại cell, dùng gallery 14C.

**Bước 1 — Median XYZ và khoảng T:** nét đứt xám là biên T V1; hai đường
T-on/T-off V2 là xanh lam/đỏ. Vùng tô là khoảng cắt thực tế của mỗi cách.
Trục thời gian là `(sample−anchor)×1000/fs`; chỉ số sample thuộc median,
không phải recording raw. QRS-on/off vẫn là biên V1.

In [ ]:
from vcg_t_almeida_plots import (
    load_case as almeida_load_case, plot_median as almeida_plot_median,
    plot_wavelets as almeida_plot_wavelets, plot_single_leads as almeida_plot_single_leads,
    plot_wt_loop as almeida_plot_wt_loop, plot_tls_fit as almeida_plot_tls_fit,
    plot_derived as almeida_plot_derived, plot_iterations as almeida_plot_iterations,
    plot_t_loops as almeida_plot_t_loops)
import matplotlib.pyplot as almeida_plt
# Module vẽ batch dùng Agg; bật formatter sau import để nhúng PNG trong notebook.
from matplotlib_inline.backend_inline import configure_inline_support, set_matplotlib_formats
configure_inline_support(get_ipython(), 'module://matplotlib_inline.backend_inline')
set_matplotlib_formats('png')

ALMEIDA_RECORD_ID = 'patient104/s0306lre'
almeida_case = almeida_load_case(almeida_out, ALMEIDA_RECORD_ID)
almeida_diag = almeida_case['diag']
almeida_columns = ['record_id','label_name','split','status',
    'qrs_on_ms','qrs_off_ms','t_on_v1_ms','t_off_v1_ms','t_duration_v1_ms',
    't_on_v2_ms','t_off_v2_ms','t_duration_v2_ms','failure_reason',
    't_on_failure_reason','t_off_failure_reason']
display(almeida_results.set_index('record_id', drop=False).loc[
    [ALMEIDA_RECORD_ID], almeida_columns])
if almeida_case['median']['xyz'] is not None:
    almeida_time = almeida_case['median']['time_ms']
    almeida_window = almeida_diag.get('search_window')
    print('fs:', almeida_case['median']['fs'], '| anchor:', almeida_case['median']['anchor'])
    print('T search samples:', almeida_window,
          '| ms:', almeida_time[almeida_window] if almeida_window else 'unavailable')
    print('ML scale:', almeida_diag.get('selected_scale'),
          '| initial TLS window:', almeida_diag.get('initial_window'))
almeida_fig = almeida_plot_median(almeida_case, comparison=True)
display(almeida_fig); almeida_plt.close(almeida_fig)

### Bước 2 — Wavelet đa scale và khởi tạo riêng X/Y/Z

Bank quadratic spline không decimation được áp dụng lên cả ba trục.
Cực trị WT biểu diễn các sườn; zero crossing hỗ trợ xác định đỉnh T.
Hình đầu cho scale 3/4/5, cực trị được chọn/bỏ và ngưỡng; hình sau cho
T-on/T-peak/T-off ứng viên riêng trên X, Y, Z cùng kiểu hình thái và lý do thiếu.
Các scale là mức của bank thiết kế, không phải cửa sổ 8/16/32 ms.

Ba loại ngưỡng có vai trò khác nhau: phát hiện `0.25 × RMS(W)`, chọn sườn
`0.125 × max|W|`, và tìm biên tương đối theo biên độ sườn. Khởi tạo đơn trục
dùng tỷ lệ onset 0.25, offset 0.4; nhánh multilead offset dùng 0.25.

In [ ]:
almeida_sl_rows = []
for almeida_sl in almeida_diag.get('single_leads', []):
    almeida_sl_rows.append({k: almeida_sl.get(k) for k in [
        'axis','wavelet_scale','morphology_type','detection_status',
        't_on_candidate_sample','t_peak_candidate_sample','t_off_candidate_sample',
        'failure_reason']})
if almeida_sl_rows:
    display(pd.DataFrame(almeida_sl_rows))
else:
    print('Không có single-lead candidates:', almeida_case['row'].get('failure_reason'))
for almeida_plot in (almeida_plot_wavelets, almeida_plot_single_leads):
    almeida_fig = almeida_plot(almeida_case)
    display(almeida_fig); almeida_plt.close(almeida_fig)

### Bước 3 — WT loop 3D, hai hướng TLS và derived wavelet

WT loop là `[W_X, W_Y, W_Z]`. TLS fit đường thẳng affine trong cửa sổ W
để tìm hướng `u`; derived wavelet là `W_d = W @ u`. Centering dùng trong fit,
còn phép chiếu dùng WT gốc. Đây là TLS trên WT loop, không phải fit XYZ.

Hai nhánh T-on/T-off lặp độc lập, cập nhật cửa sổ theo biên và sườn liên quan.
Các hình dưới hiển thị vòng được chọn của từng nhánh, hướng fit, ngưỡng và
candidate. Hình lịch sử cùng bảng ghi toàn bộ vòng lặp, cửa sổ, trạng thái và
nguyên nhân dừng; gallery 14C cho phép chọn xem bất kỳ vòng nào.
Khi nhánh không giải được, hình và bảng giữ nguyên lý do thiếu.

In [ ]:
almeida_fig = almeida_plot_wt_loop(almeida_case)
display(almeida_fig); almeida_plt.close(almeida_fig)
for almeida_boundary in ('t_on', 't_off'):
    almeida_branch = almeida_diag.get('branches', {}).get(almeida_boundary, {})
    display(Markdown('**'+almeida_boundary+'** — chosen iteration: '+
        str(almeida_branch.get('chosen_iteration'))+'; stop: '+
        str(almeida_branch.get('stop_reason', 'unavailable'))))
    almeida_fig = almeida_plot_tls_fit(almeida_case, boundary=almeida_boundary)
    display(almeida_fig); almeida_plt.close(almeida_fig)
    almeida_fig = almeida_plot_derived(almeida_case, boundary=almeida_boundary)
    display(almeida_fig); almeida_plt.close(almeida_fig)
almeida_history = []
for almeida_boundary, almeida_branch in almeida_diag.get('branches', {}).items():
    for almeida_entry in almeida_branch.get('history', []):
        almeida_history.append({k: almeida_entry.get(k) for k in [
            'boundary','iteration','window_start','window_end','direction',
            'fit_quality','extremum_sample','extremum_amplitude','boundary_threshold',
            'candidate_sample','accepted','status','stop_reason']})
display(pd.DataFrame(almeida_history))
almeida_fig = almeida_plot_iterations(almeida_case)
display(almeida_fig); almeida_plt.close(almeida_fig)

### Bước 4 — T loops V1/V2 trên chính XYZ của median

Sau khi có đủ hai biên hợp lệ, loop được cắt bằng `xyz[on:off+1]` trên median
V1. Hai hình 3D dùng chung gốc PQ và tỉ lệ; ba hình chiếu XY/XZ/YZ giữ tỉ lệ
bằng nhau. Chấm tròn là đầu, dấu x là cuối; mũi tên chỉ chiều thời gian.
Nhánh thiếu một biên không tạo loop. QRS vẫn dùng khoảng V1 đã lưu.

In [ ]:
almeida_fig = almeida_plot_t_loops(almeida_case)
display(almeida_fig); almeida_plt.close(almeida_fig)

## 14E. Các ca cần xem lại: đủ biên, T ngắn, thiếu biên và thiếu median

Bốn ID dưới giống các ví dụ V1 trong Notebook 08 và Notebook 09. Bảng giữ
cả biên từng phần và nguyên nhân thất bại; các hình so sánh bên dưới có cả
ca unresolved. `NaN` là thiếu kết quả, không phải 0 ms.

Đặc biệt, `patient051/s0179lre` có T V1 dài 17 ms; V2 mới tìm được T-on,
chưa có T-off nên **chưa có T loop V2**. `patient034/s0109lre` vẫn chưa giải
được cặp biên T; `patient033/s0157lre` thiếu median từ bước trước.

In [ ]:
almeida_example_ids = ['patient104/s0306lre','patient051/s0179lre',
                       'patient034/s0109lre','patient033/s0157lre']
display(almeida_results.set_index('record_id', drop=False).loc[
    almeida_example_ids, almeida_columns])
for almeida_rid in almeida_example_ids:
    display(Markdown('**'+almeida_rid+' — XYZ/biên T và T loops V1/V2**'))
    for almeida_kind in ('comparison', 'loops'):
        almeida_png = almeida_out/'figures'/(almeida_rid.replace('/','_')+'_'+almeida_kind+'.png')
        if not almeida_png.is_file():
            raise FileNotFoundError('Thiếu hình đã lưu: '+str(almeida_png))
        display(Image(filename=str(almeida_png), width=1150))
display(Markdown(
    '**Kết quả cấu hình hiện tại:** '+str(almeida_summary['t_loops_available'])+
    ' T loops V2 / '+str(almeida_summary['median_available'])+' median khả dụng; '+
    str(almeida_summary['statuses'].get('unresolved', 0))+
    ' ID unresolved trên tổng '+str(len(almeida_results))+' ID. '
    'Coverage này không phải tỷ lệ tách đúng. '
    '[Xem đủ 448 ca trong gallery](outputs/vcg_t_almeida_v2_experiment/gallery/index.html) · '
    '[Báo cáo Wavelet/TLS](outputs/vcg_t_almeida_v2_experiment/reports/experiment_report.md).'))

<a id="nb08-all-t-loops"></a>

## 14F. Tất cả 448 bản ghi: XYZ, biên T và T loops V1/V2 ngay trong notebook

**Mọi ID trong manifest xuất hiện đúng một lần trong các trang hình dưới đây.**
Mỗi dòng là một bản ghi: bên trái là median XYZ và các biên T đã lưu; giữa là
T loop V1; bên phải là T loop Wavelet/TLS V2. Hai loop 3D dùng cùng gốc PQ và
cùng tỉ lệ trong từng bản ghi, có dấu đầu/cuối và mũi tên thời gian.

Ca chưa có đủ hai biên T hoặc thiếu median vẫn có dòng riêng ghi rõ nguyên
nhân. Biên T từng phần vẫn hiện trên XYZ. Không lọc theo QC, nhãn, split hoặc
độ đẹp của loop. Các trang sắp theo record ID; 6 bản ghi/trang, trang cuối 4.

Hình được nhúng sẵn để đọc trực tiếp. Dưới mỗi tiêu đề trang có liên kết tới
hình T loops đầy đủ của từng ID (kèm ba hình chiếu XY/XZ/YZ). Để xem wavelet,
từng vòng TLS, zoom hoặc xoay 3D của bất kỳ ca nào, dùng gallery 14C.

In [ ]:
from vcg_t_almeida_overview import create_t_loop_contact_sheets
almeida_all_loop_out = almeida_out/'notebook08_extension/all_record_t_loops'
almeida_all_loop_manifest, almeida_all_loop_summary = create_t_loop_contact_sheets(
    almeida_results, almeida_out, almeida_all_loop_out, page_size=6)
if (len(almeida_all_loop_manifest) != 448
        or not almeida_all_loop_manifest.record_id.is_unique
        or set(almeida_all_loop_manifest.record_id) != set(almeida_manifest.record_id)):
    raise ValueError('Hình tổng quan T loops chưa bao phủ đúng tất cả 448 ID.')
display(pd.Series(almeida_all_loop_summary, name='Hiển thị toàn bộ T loops').to_frame())
display(Markdown(
    '[Mở toàn bộ các trang T loops trong tab riêng]'
    '(outputs/vcg_t_almeida_v2_experiment/notebook08_extension/all_record_t_loops/index.html). '
    'Mỗi ID có một dòng trong manifest.csv của thư mục này.'))
for almeida_page, almeida_page_rows in almeida_all_loop_manifest.groupby('page', sort=True):
    display(Markdown('### Trang '+str(almeida_page)+'/'+
        str(almeida_all_loop_summary['pages'])+' — '+str(len(almeida_page_rows))+' bản ghi'))
    display(Markdown(' · '.join(
        '['+almeida_rid+'](outputs/vcg_t_almeida_v2_experiment/figures/'+
        almeida_rid.replace('/','_')+'_loops.png)'
        for almeida_rid in almeida_page_rows.record_id)))
    display(Image(filename=str(almeida_all_loop_out/almeida_page_rows.plot_path.iloc[0]),
                  width=1700))

## 3. Cách dùng Manual Visual Review và đọc chênh lệch

Các mốc thủ công là đánh giá bằng mắt, không tự biến thành ground truth độc lập. Dữ liệu review nằm trong trình duyệt và CSV do người dùng xuất; Run All không ghi đè đánh dấu.

In [ ]:
display(Markdown(
    '1. Mở gallery, nhập tên/mã reviewer rồi chọn recording. Có thể tắt bốn mốc V1 để quan sát signal; '
    'việc đã từng thấy prediction vẫn được ghi lại.\n\n'
    '2. Zoom vùng QRS/T, chọn biên đang đặt rồi click trên signal hoặc nhập thời gian ms. '
    'Bật overlay để so mốc thủ công với V1; không thay mốc đã lưu của pipeline.\n\n'
    '3. Chọn nhận xét: đạt, quá sớm, quá muộn, không rõ biên hoặc không đánh giá được; ghi chú rồi lưu. '
    'Chuyển Previous/Next và quay lại vẫn giữ đánh dấu. Xuất CSV thường xuyên để sao lưu.\n\n'
    '**Chênh lệch = V1 time − Manual time.** Dương: V1 muộn hơn mốc bạn chọn; âm: V1 sớm hơn. '
    'Nếu thiếu một mốc thì không tính chênh lệch, không thay bằng0.\n\n'
    '**Blind/unblind:** ẩn prediction sau khi đã thấy không làm review trở lại độc lập. '
    'CSV này luôn là visual_review_only; muốn nghiên cứu accuracy, dùng quy trình annotation độc lập ở phụ lục.'))

## 4. Ví dụ trực tiếp: đủ biên, T rất ngắn, thiếu biên và thiếu median

Các hình dưới minh họa chính signal và mốc V1 đã lưu. Không gán đúng/sai chỉ từ độ rộng, ST hoặc nhãn MI. Trong gallery, mọi recording đều có cùng cách xem.

In [ ]:
for rid in ['patient104/s0306lre','patient051/s0179lre','patient034/s0109lre','patient033/s0157lre']:
    row=frame.set_index('record_id').loc[rid]
    display(Markdown(f"**{rid} — {row.label_name}, {row.split}, QC={row.qc_status}**"))
    display(row[['qrs_on_ms','qrs_off_ms','t_on_ms','t_peak_ms','t_off_ms']].to_frame('V1_ms'))
    image_row=panel_manifest.set_index('record_id').loc[rid]
    display(Image(filename=str(REVIEW_OUT/image_row.plot_path),width=1250))

## 14B. Tổng quan đủ448 median XYZ và các vùng V1 đã cắt

Mỗi ca xuất hiện đúng một lần trên các trang hình. XYZ ngoài khoảng QRS/T vẫn hiện; ca không có median hiển thị nguyên nhân. Đây là hình tổng quan; mở gallery14A để zoom và đặt mốc.

In [ ]:
contact_manifest = create_v1_contact_sheets(frame,ARTIFACTS,REVIEW_OUT,page_size=12)
assert len(contact_manifest)==448 and contact_manifest.record_id.is_unique
print('Toàn bộ',len(contact_manifest),'ca /',contact_manifest.page.nunique(),'trang XYZ')
for page,rows in contact_manifest.groupby('page',sort=True):
    display(Markdown(f'**Trang {page}/{contact_manifest.page.nunique()}**'))
    display(Image(filename=str(REVIEW_OUT/rows.plot_path.iloc[0]),width=1400))

## 5. Tóm tắt kiểm tra trực quan và các ca thiếu/lỗi

Báo số lượng khả dụng và khả năng mở hình; không báo tỷ lệ phân đoạn đúng khi chưa có tham chiếu. Mọi lỗi dữ liệu/render vẫn có dòng trong manifest và thông báo ở giao diện.

In [ ]:
display(signal_manifest.groupby('segmentation_status').size().rename('recordings').to_frame())
display(signal_manifest[signal_manifest.segmentation_status.ne('complete_bounds')].head(30))
visual_checks = {'records':len(frame),'median_records':int(signal_manifest.has_median.sum()),
    'panel_records':len(panel_manifest),'contact_sheet_records':len(contact_manifest),
    'contact_sheet_pages':contact_manifest.page.nunique(),
    'manual_reviews_independent_reference':False,'test_used_for_tuning':False}
write_json(REVIEW_OUT/'visual_review_execution_checks.json',visual_checks)
print('Các đánh dấu được export riêng; không sửa reference_annotations.csv hoặc prediction V1.')

# Phụ lục — Reference và sai số chỉ khi có annotation độc lập

Phần dưới giữ quy trình đánh giá cũ để bổ sung bằng chứng khi có tham chiếu hợp lệ.
Dữ liệu Manual Visual Review ở giao diện chính không tự được nhập làm reference.
Hai candidate gần nhau vẫn có thể cùng sai, nên đồng thuận không được coi là accuracy.

## A1. Kiểm tra nguồn tham chiếu thực tế

Xác minh nguồn, record, signal/lead và hệ tọa độ thời gian.

In [ ]:
sources, candidates, headers = scan_reference_sources(frame, OUT)
display(headers.groupby(['fs','frank_channels','frank_units']).size().rename('recordings').to_frame())
display(pd.read_csv(OUT/'local_file_extension_inventory.csv'))
display(candidates[['path','type','n_rows','finite_reference_values','status']])
display(sources[['source_type','annotation_scope','status','reason']])
display(Markdown('**Kết luận nguồn:** `.xyz` là file waveform Frank được header trỏ tới, không phải annotation. Các file template không có giá trị ref không tạo ground truth. Xem reference_sources.json để truy nguồn tài liệu.'))

## A2. Hệ thời gian của median beat

Giữ đúng fs/anchor và cửa sổ signal; không so raw với median khi chưa có ánh xạ hợp lệ.

In [ ]:
demo_id = frame.loc[frame.processing_success, 'record_id'].iloc[0]
demo = load_median(ARTIFACTS, demo_id)
print('Record minh họa:', demo_id, '| fs:', demo['fs'])
print('Median signal SHA-256:', demo['signal_sha256'])
display(Markdown(
    '**Thời gian:** t[i]=(i−median_anchor_sample)×1000/fs. Reference được gắn vào mẫu gần nhất, '
    'tie chọn mẫu sớm; ghi riêng quantization_error_ms. Ngoài cửa sổ bị từ chối, không clamp. '
    '**Khoảng đo:** duration=t_off−t_on theo thời gian liên tục; cắt XYZ/features bằng slice inclusive đúng V1. '
    'Độ dài cắt n_samples không được thay thế duration bằng n_samples/fs.'))

## A3. Danh sách ưu tiên review trước đây

Chỉ là gợi ý; gallery chính vẫn hiển thị mọi recording.

In [ ]:
priority = priority_cases(frame, OUT)
reason_counts = priority.priority_reasons.str.get_dummies(';').sum().sort_values(ascending=False)
display(reason_counts.rename('recordings').to_frame())
display(priority.loc[priority.eligible_for_development_review,
    ['record_id','label_name','split','priority_reasons','t_duration_ms','t_on_method_gap_ms','t_off_method_gap_ms']].head(18))
display(Markdown('**Giới hạn:** bất đồng phương pháp không phải actual error. Theo yêu cầu xem toàn bộ, gallery riêng ở mục14A hiển thị cả test để mô tả; không dùng các hình test để điều chỉnh thuật toán hoặc chọn ngưỡng.'))

## A4. Công cụ annotation độc lập riêng

Dành cho người đánh giá chưa xem prediction; tách khỏi Manual Visual Review ở gallery chính.

In [ ]:
selected07 = pd.read_csv(ROOT/'outputs/vcg_qc_audit_v1/manual_review_blinded.csv')
REVIEW_IDS = list(selected07.record_id)
assert len(REVIEW_IDS)>=63 and set(REVIEW_IDS).issubset(set(frame[frame.split.ne('test')].record_id))
review_records = frame.set_index('record_id').loc[REVIEW_IDS].reset_index()
ui = build_annotation_tool(frame, review_records, ARTIFACTS, OUT, cfg=CFG)
print('Công cụ:', ui['html_path'])
print('Review records:', len(REVIEW_IDS), '| Không có median V1:', int((~review_records.processing_success).sum()))
display(review_records.groupby(['split','label_name']).size().rename('selected_recordings').to_frame())
display(Markdown('[Mở công cụ annotation trong tab riêng](outputs/vcg_boundary_reference_eval_v1/review/independent_annotation.html). '
    'Trong Jupyter có thể dùng iframe dưới đây. File CSV được lưu/export bằng thao tác của reviewer, chưa có mốc nào được điền tự động.'))
display(IFrame(src='outputs/vcg_boundary_reference_eval_v1/review/independent_annotation.html', width='100%', height=1000))

## A5. Đọc reference đã được cung cấp

Không ghi đè annotation hiện có và không tự nâng trạng thái chấp nhận.

In [ ]:
annotations = read_annotations(OUT, ANNOTATION_FILES)
imported07 = import_notebook07_reviews(frame, ARTIFACTS, OUT)
if not imported07.empty:
    annotations = pd.concat([annotations, imported07], ignore_index=True)
print('Số hàng annotation/template đã đọc:', len(annotations))
print('Số hàng có giá trị review thực từ notebook07:', len(imported07))
display(annotations[['record_id','reviewer','annotation_scope','reference_quality','review_status',
    'qrs_on_ref_ms','qrs_off_ref_ms','t_on_ref_ms','t_off_ref_ms']].head())
display(Markdown('**Điều kiện:** confirmed + chất lượng good/accepted + độc lập rõ ràng + đúng record/patient/split/fs/hash/median scope. '
    'Một biên uncertain/not_visible/rejected không được dùng tính error. Hai reviewer trùng cần primary/adjudicated rõ ràng; không chọn ngầm.'))

## A6. Xác minh reference và tính bảng sai số

Chỉ chấp nhận reference độc lập đúng identity/time/hash/scope.

In [ ]:
results = evaluate_references(frame, annotations, ARTIFACTS, CFG, allow_test=FINAL_TEST_EVALUATION)
ledger, status = save_results(results, frame, OUT, allow_test=FINAL_TEST_EVALUATION)
display(pd.DataFrame([status]))
display(results['annotation_acceptance_audit'].head(20))
display(ledger.reference_status.value_counts().rename('recordings').to_frame())
print('Trạng thái hiện tại:', status['mode'])

## A7. Sai số từng biên và coverage

Thiếu reference/prediction được giữ rõ; không thay sai số thiếu bằng0.

In [ ]:
boundary = results['boundary_error_per_record']
summary = results['boundary_error_summary']
display(summary[summary.group_by.isin(['all','train_valid','label_name','split'])])
display(boundary[['record_id','boundary','predicted_ms','reference_ms','error_ms',
    'absolute_error_ms','reference_available','prediction_available','reference_status']].head(12))
if not boundary.error_ms.notna().any():
    display(Markdown('**GROUND_TRUTH_UNAVAILABLE:** chưa thể trả lời bốn mốc lệch bao nhiêu ms; NaN không phải sai số0.'))

## A8. Theo bệnh nhân và giữa reviewer

Không coi recordings cùng bệnh nhân là độc lập.

In [ ]:
display(results['patient_boundary_summary'].head(20))
display(results['patient_feature_summary'].head(20))
display(results['interreviewer_disagreement'].head(20))
display(Markdown('**Phạm vi suy luận:** các recordings cùng người không độc lập. Mẫu review được ưu tiên không cho ước lượng '
    'sai số toàn448 nếu chưa có kế hoạch lấy mẫu xác suất hoặc annotation đủ rộng. Các bảng record là mô tả có điều kiện trên reference hiện có.'))

## A9. Sai lệch khoảng cắt QRS/T

IoU và cắt thừa/thiếu chỉ tính khi có đủ cặp khoảng hợp lệ.

In [ ]:
loops = results['loop_overlap_per_record']
display(results['loop_overlap_summary'])
display(loops.head(12))
display(Markdown('**Định nghĩa:** IoU=intersection/union; missed=reference_duration−intersection; '
    'excess=predicted_duration−intersection; reference coverage=intersection/reference_duration. '
    'duration_error=pred_duration−ref_duration. Nếu một khoảng chưa có, metric chưa thể tính.'))

## A10. Sai lệch12 features theo reference

Giữ đúng công thức và vận tốc trên toàn median của V1.

In [ ]:
features = results['feature_reference_comparison']
assert set(features.feature)==set(FEATURE_NAMES)
display(results['baseline_reproduction_check'].head(15))
display(results['feature_error_summary'].head(24))
display(features.head(12))
display(Markdown('**Công thức:** feature_error=pred−ref; absolute=abs(pred−ref); relative_pct=100×absolute/abs(ref). '
    'Khi abs(ref)<=1e−9 trong đơn vị riêng của feature: relative=NaN và bật near_zero. '
    'Bảng này đo thay đổi do biên; không đo khả năng phân loại bệnh và không phải feature importance.'))

## A11. Toàn bộ hình đối chiếu có reference

Nếu có reference, vẽ mọi ca hợp lệ; chưa có thì không tạo hình accuracy giả.

In [ ]:
plotted = create_reference_plots(results, ARTIFACTS, OUT, cfg=CFG)
display(plotted['manifest'])
manifest_plot = plotted['manifest']
# Chỉ hiển thị hình có trạng thái created của chính lần đánh giá hiện tại.
for plot_row in manifest_plot.itertuples(index=False):
    if plot_row.status=='created' and str(plot_row.plot_path).endswith('.png'):
        display(Image(filename=str(OUT/plot_row.plot_path), width=1050))
if plotted['gallery'] is not None:
    display(Markdown('[Gallery overlay QRS/T có reference](outputs/vcg_boundary_reference_eval_v1/reference_loop_gallery.html).'))
if plotted.get('all_gallery') is not None:
    display(Markdown('[Tất cả loop có reference hợp lệ](outputs/vcg_boundary_reference_eval_v1/all_reference_loop_gallery.html).'))
if plotted.get('boundary_gallery') is not None:
    display(Markdown('[Tất cả biên có reference hợp lệ, kể cả reference từng phần](outputs/vcg_boundary_reference_eval_v1/all_reference_boundary_gallery.html).'))
if status['n_valid_boundary_references']==0:
    display(Markdown('**Chưa có error plots hoặc gallery tốt/xấu:** không có cặp reference hợp lệ để xếp hạng. Xem plots_status.csv.'))

## A12. Báo cáo11 câu hỏi reference

Kết luận định lượng theo dữ liệu thật, không theo mức đồng thuận candidate.

In [ ]:
priority_current = pd.read_csv(OUT/'priority_review_cases.csv')
report_path, conclusions = write_report(results, frame, status, priority_current, OUT)
for i, item in enumerate(conclusions, 1):
    display(Markdown(f"**{i}. {item['question']}** {item['answer']}"))
print('Báo cáo:', report_path)

## B. Chẩn đoán tangent/threshold tùy chọn

Đã chuyển khỏi nội dung xem chính. Mặc định không tạo thêm biểu đồ đồng thuận.

In [ ]:
if SHOW_METHOD_DIAGNOSTICS:
    from vcg_all_case_visual_audit import build_disagreement_tables,plot_disagreement_summary
    method_tables=build_disagreement_tables(frame,OUT)
    for item in plot_disagreement_summary(method_tables,OUT).itertuples(index=False):
        display(Image(filename=str(OUT/item.plot_path),width=1100))
else:
    display(Markdown('Phân tích candidate đang tắt. Bật SHOW_METHOD_DIAGNOSTICS nếu cần; '
        'hai phương pháp đồng thuận không chứng minh biên đúng.'))

## 6. Kiểm tra cuối và nguồn tái lập

Đối chiếu hash dữ liệu/baseline; kiểm tra manifest448 ca, mọi hình và snapshot mã nguồn. Không sửa baseline hoặc tự tạo reference.

In [ ]:
assert verify_protected(PROTECTED)
assert set(signal_manifest.record_id)==set(frame.record_id)==set(panel_manifest.record_id)==set(contact_manifest.record_id)
assert all((REVIEW_OUT/p).is_file() for p in panel_manifest.plot_path)
assert all((REVIEW_OUT/p).is_file() for p in contact_manifest.plot_path.unique())
assert (REVIEW_OUT/'all_cases_gallery.html').is_file()
source_snapshot=snapshot_sources(OUT)
write_json(OUT/'notebook_execution_checks.json',{
    **visual_checks,'mode':'VISUAL_V1_REVIEW_WITH_OPTIONAL_REFERENCE',
    'saved_predictions_used':True,'reference_mode':status['mode'],
    'accepted_boundary_references':status['n_valid_boundary_references'],
    'baseline_algorithm_modified':False,'protected_inputs_unchanged_during_final_execution':True,
    'source_snapshot_files':len(source_snapshot)})
display(Markdown('**Hoàn tất:** [Gallery tương tác448 ca](outputs/vcg_boundary_reference_eval_v1/all_cases/all_cases_gallery.html) · '
    '[Báo cáo reference bổ sung](outputs/vcg_boundary_reference_eval_v1/reference_evaluation_report.md). '
    'Luồng chính dùng signal và mốc đã lưu; mốc thủ công luôn tách khỏi V1 và annotation độc lập.'))